# Projeto Prático 1 (Parte 2) — Comparativo Industrial: `Scikit-Learn` e `Keras`

Na **[Parte 1 (`02_projeto1_pmc_regressao_numpy.ipynb`)](02_projeto1_pmc_regressao_numpy.ipynb)** construímos todas as equações matriciais do PMC `3-10-1` do zero em `NumPy`.

Agora responderemos à pergunta prática de todo engenheiro: **como implementamos esse mesmo aproximador de funções no dia a dia da indústria usando `Scikit-Learn` e `TensorFlow/Keras`?**

---

## 1. Carregando os Mesmos Dados de Treino e Teste

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

train_df = pd.read_csv("datasets/projeto-pratico-1.csv")
test_df = pd.read_csv("datasets/conjunto-testes-projeto-pratico-1.csv")

X_train = train_df[["x1", "x2", "x3"]].to_numpy()
D_train = train_df["d"].to_numpy()  # Vetor 1D para compatibilidade direta com Scikit-Learn

X_test = test_df[["x1", "x2", "x3"]].to_numpy()
D_test = test_df["d"].to_numpy()

print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")

---

## 2. Abordagem 1: `MLPRegressor` (`Scikit-Learn`)

O `MLPRegressor` cria uma rede de regressão em poucas linhas.
> **Atenção a um detalhe técnico crucial:** Na camada de saída, o `MLPRegressor` usa função de ativação **identidade/linear** ($y = u$), enquanto na camada oculta podemos escolher `'logistic'` (sigmóide) ou `'relu'`. Além disso, para tabelas compactas (`200` amostras), o otimizador determinístico de segunda ordem `solver='lbfgs'` converge em frações de segundo com altíssima precisão! Vamos comparar `sgd` (gradiente estocástico clássico) e `lbfgs`:

In [ ]:
from sklearn.neural_network import MLPRegressor

resultados_sk = []
for i in range(5):
    # Configura topologia (10,) = 1 camada oculta com 10 neurônios e ativação logística
    sk_mlp = MLPRegressor(
        hidden_layer_sizes=(10,),
        activation="logistic",
        solver="lbfgs",        # Método quase-Newton (equivalente industrial ao Levenberg-Marquardt!)
        max_iter=2000,
        random_state=42 + i,
    )
    sk_mlp.fit(X_train, D_train)
    pred_sk = sk_mlp.predict(X_test)

    erros_rel = (np.abs(D_test - pred_sk) / D_test) * 100.0
    resultados_sk.append({
        "Modelo": f"SK_LBFGS_{i + 1}",
        "Iterações": sk_mlp.n_iter_,
        "Loss Final": sk_mlp.loss_,
        "Erro Relativo Médio (%)": np.mean(erros_rel),
        "Variância (%)": np.var(erros_rel),
    })

df_sk = pd.DataFrame(resultados_sk)
print("Resultados no Teste com Scikit-Learn (MLPRegressor + lbfgs):")
print(df_sk.round(4).to_string(index=False))

**Observação:** Veja como o otimizador de segunda ordem (`lbfgs`) derrubou o erro relativo médio em poucas centenas de iterações!

---

## 3. Abordagem 2: Deep Learning com `Keras` (`TensorFlow`)

No **`Keras`**, temos liberdade total para reproduzir **exatamente** a saída sigmóide `Dense(1, activation='sigmoid')` do projeto original e usar o otimizador adaptativo **`Adam`** com *Mini-Batches* (`batch_size=16`).

### Por que o `batch_size` faz tanta diferença?
- **Batch Completo (`batch_size = 200`):** Atualiza os pesos apenas **1 vez por época** (em 1000 épocas = 1000 ajustes).
- **Mini-Batch (`batch_size = 16`):** Divide as 200 amostras em pacotes de 16, fazendo **13 atualizações por época** com vetorização rápida de GPU/CPU!

> **Boa Prática Keras 3:** Declaramos sempre `keras.Input(shape=(3,))` como primeira camada explícita de entrada (evitando o antigo `input_shape` dentro de `Dense`, que hoje gera `UserWarning`).

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Fixa semente global do TensorFlow para reprodutibilidade
tf.random.set_seed(42)

resultados_keras = []
for i in range(3):
    # Arquitetura idêntica 3 -> 10 (sigmoid) -> 1 (sigmoid) usando keras.Input moderno
    modelo_k = keras.Sequential([
        keras.Input(shape=(3,)),
        keras.layers.Dense(10, activation="sigmoid"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])

    modelo_k.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.01),
        loss="mse",
    )

    # Treina silenciosamente (verbose=0) em mini-batches de 16 amostras
    hist_k = modelo_k.fit(X_train, D_train, epochs=1500, batch_size=16, verbose=0)

    # Chamada direta modelo_k(X_test, training=False) evita avisos de retracing em loops!
    pred_k = modelo_k(X_test, training=False).numpy().ravel()
    erros_rel_k = (np.abs(D_test - pred_k) / D_test) * 100.0

    resultados_keras.append({
        "Modelo": f"Keras_Adam_{i + 1}",
        "MSE Treino Final": hist_k.history["loss"][-1],
        "Erro Relativo Médio (%)": np.mean(erros_rel_k),
        "Variância (%)": np.var(erros_rel_k),
    })

df_keras = pd.DataFrame(resultados_keras)
print("Resultados no Teste com TensorFlow/Keras (Adam + Mini-Batch):")
print(df_keras.round(5).to_string(index=False))

## Quadro Comparativo para o Engenheiro

| Ferramenta | Quando Usar? | Vantagem Principal |
| :--- | :--- | :--- |
| **`NumPy` Puro** | Estudo acadêmico, sistemas embarcados leves (C/microcontroladores) | Controle 100% transparente de cada equação matricial. |
| **`Scikit-Learn`** | Dados tabulares industriais pequenos e médios (`N < 50.000`) | Rápido, estável (`lbfgs`) e sem configuração pesada. |
| **`Keras / TensorFlow`** | Grandes volumes de dados, imagens, séries longas e arquiteturas customizadas | Suporte a GPU, camadas variadas e otimizadores modernos (`Adam`). |

---

## 4. Exercícios Práticos de Fixação

### Exercício 1: Trocando Sigmóide por `ReLU` no `MLPRegressor`
Altere `activation='relu'` no `MLPRegressor` do `Scikit-Learn` mantendo `hidden_layer_sizes=(10,)`, `solver='lbfgs'` e `random_state=42`. Calcule o Erro Relativo Médio (%) sobre `X_test` e veja se a função `ReLU` compete bem com a sigmóide!

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
from sklearn.neural_network import MLPRegressor

mlp_relu = MLPRegressor(hidden_layer_sizes=(10,), activation="relu", solver="lbfgs", max_iter=2000, random_state=42)
mlp_relu.fit(X_train, D_train)
pred_relu = mlp_relu.predict(X_test)

erro_medio_relu = np.mean(np.abs(D_test - pred_relu) / D_test * 100.0)
print(f"Erro Relativo Médio com ativação ReLU: {erro_medio_relu:.4f}%")
```

</details>

### Exercício 2: Efeito do `batch_size` Grande no `Keras`
Treine um modelo `Keras` idêntico por `500` épocas usando `batch_size=200` (lote inteiro, apenas 1 ajuste por época) vs. `batch_size=16` e compare o `loss` (`MSE`) final atingido em cada caso.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
for tamanho_lote in [200, 16]:
    tf.random.set_seed(42)
    m = keras.Sequential([
        keras.Input(shape=(3,)),
        keras.layers.Dense(10, activation="sigmoid"),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01), loss="mse")
    h = m.fit(X_train, D_train, epochs=500, batch_size=tamanho_lote, verbose=0)
    print(f"batch_size = {tamanho_lote:3d}  -->  MSE após 500 épocas: {h.history['loss'][-1]:.6f}")
```

</details>